# Merge syst dfs from grid job

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

import uproot
from matplotlib import gridspec

import sys
sys.path.append('../../../../')
from pyanalib.split_df_helpers import *
import pyanalib.pandas_helpers as ph
import pyanalib.stat_helpers as sh
from makedf.util import *
from analysis_village.plot_style.plot_helper import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

In [3]:
sample_str = "v10_14_02_02+"

## Open files and concat

In [4]:
input_path = "/data/sungbino/sbnd/gen2/cohpi/syst_no_dirz_10k/"
prefix = "cohpi_all_weight_signal_mc_no_dirz_"

### Test 1 file

In [5]:
print_keys(input_path + prefix + "103.df")

Keys: ['/evt_0', '/hdr_0', '/histgenevtdf_0', '/histpotdf_0', '/mcnu_0', '/pot_0', '/split']


In [6]:
keys2load = ['hdr', "mcnu", 'evt', "pot", "histpotdf", "histgenevtdf"]

In [7]:
test_dfs = load_dfs(input_path + prefix + "103.df", keys2load, n_max_concat=1)

In [8]:
test_dfs['mcnu']

true_E_nu true_p_mu true_p_pi  \
                                                                
__ntuple entry rec.mc.nu..index                                 
24       1     0                 0.811776  0.238291  0.533144   

                                true_cos_theta_mu true_cos_theta_pi    true_t  \
                                                                                
__ntuple entry rec.mc.nu..index                                                 
24       1     0                         0.973984          0.996001  0.004812   

                                nuint_categ ind  \
                                                  
__ntuple entry rec.mc.nu..index                   
24       1     0                          1   0   

                                GENIEReWeight_SBN_v1_multisigma_ZExpA1CCQE  \
                                                                       ps1   
__ntuple entry rec.mc.nu..index                                              
24       1     0                                                       1.0   

                                      ... reinteractions_proton_Geant4  \
                                 ms1  ...                     univ_990   
__ntuple entry rec.mc.nu..index       ...                                
24       1     0                 1.0  ...                     1.000826   

                                                                       \
                                 univ_991  univ_992 univ_993 univ_994   
__ntuple entry rec.mc.nu..index                                         
24       1     0                 1.000247  0.998392  0.99928  1.00052   

                                                                        \
                                 univ_995  univ_996 univ_997  univ_998   
__ntuple entry rec.mc.nu..index                                          
24       1     0                 1.000808  1.000958  0.99843  0.996657   

                                           
                                 univ_999  
__ntuple entry rec.mc.nu..index            
24       1     0                 1.000579  

[1 rows x 22573 columns]

In [9]:
test_dfs['evt'].index = test_dfs['evt'].index.droplevel(1)

# 2. Move 'entry' and 'rec.slc..index' into the index (appending to '__ntuple')
test_dfs['evt'] = test_dfs['evt'].set_index(['entry', 'rec.slc..index'], append=True)

In [10]:
test_dfs['evt']

is_fv bcfm_score  nu_score is_not_clear_cosmic  \
                                                                               
__ntuple entry rec.slc..index                                                  
24       1     1               True   0.113778  0.652827                True   
25       8     0               True   0.102986  0.464283                True   
54       1     0               True   0.102739  0.664467                True   

                              n_prong n_trk n_proton n_shower tmatch_idx  \
                                                                           
__ntuple entry rec.slc..index                                              
24       1     1                    2     2        0        0          0   
25       8     0                    2     2        0        0       -999   
54       1     0                    2     2        0        0          0   

                              tmatch_eff  ... reinteractions_proton_Geant4  \
                                          ...                     univ_990   
__ntuple entry rec.slc..index             ...                                
24       1     1                0.761866  ...                     1.000826   
25       8     0                     NaN  ...                          NaN   
54       1     0                0.964623  ...                     1.015296   

                                                                       \
                               univ_991  univ_992  univ_993  univ_994   
__ntuple entry rec.slc..index                                           
24       1     1               1.000247  0.998392  0.999280  1.000520   
25       8     0                    NaN       NaN       NaN       NaN   
54       1     0               1.004540  0.970848  0.986858  1.009609   

                                                                       \
                               univ_995  univ_996  univ_997  univ_998   
__ntuple entry rec.slc..index                                           
24       1     1               1.000808  1.000958  0.998430  0.996657   
25       8     0                    NaN       NaN       NaN       NaN   
54       1     0               1.014959  1.017759  0.971542  0.940305   

                                         
                               univ_999  
__ntuple entry rec.slc..index            
24       1     1               1.000579  
25       8     0                    NaN  
54       1     0               1.010692  

[3 rows x 22620 columns]

### Concat all

In [11]:
from tqdm import tqdm
import pandas as pd
import warnings
#warnings.simplefilter("ignore", category=FutureWarning)

# 1. Initialize empty lists to hold the dataframes and indices
evt_list = []
mcnu_list = []
pot_list = []
hdr_list = []
histpotdf_list = []
histgenevtdf_list = []
file_indices = [] # NEW: Keep track of 'i' for the concatenation

for i in tqdm(range(0, 10000), desc="Loading DataFrames"):
    # Load data
    test_dfs = load_dfs(input_path + prefix + f"{i}.df", keys2load, n_max_concat=1)

    # Process the evt DataFrame
    this_evt_df = test_dfs['evt']
    this_evt_df.index = this_evt_df.index.droplevel(1)
    this_evt_df = this_evt_df.set_index(['entry', 'rec.slc..index'], append=True)

    # 2. Append to lists INSTEAD of concatenating in the loop
    evt_list.append(this_evt_df)
    mcnu_list.append(test_dfs['mcnu'])
    pot_list.append(test_dfs['pot'])
    hdr_list.append(test_dfs['hdr'])
    histpotdf_list.append(test_dfs['histpotdf'])
    histgenevtdf_list.append(test_dfs['histgenevtdf'])
    
    file_indices.append(i) # NEW: Store the loop index

# 3. Concatenate everything ONCE at the end using the 'keys' argument.
# This automatically prepends 'i' as an outermost index level called 'file_idx'.
evt_df = pd.concat(evt_list, keys=file_indices, names=['file_idx']) 
mcnu_df = pd.concat(mcnu_list, keys=file_indices, names=['file_idx']) 
pot_df = pd.concat(pot_list, keys=file_indices, names=['file_idx'])
hdr_df = pd.concat(hdr_list, keys=file_indices, names=['file_idx'])
histpotdf_df = pd.concat(histpotdf_list, keys=file_indices, names=['file_idx'])
histgenevtdf_df = pd.concat(histgenevtdf_list, keys=file_indices, names=['file_idx'])

Loading DataFrames: 100%|██████████| 10000/10000 [12:02<00:00, 13.84it/s]
/tmp/ipykernel_17738/2891820499.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  evt_df = pd.concat(evt_list, keys=file_indices, names=['file_idx'])


In [12]:
evt_df

is_fv bcfm_score  nu_score  \
                                                                    
file_idx __ntuple entry rec.slc..index                              
0        3        4     3               True   0.277667  0.717133   
         36       6     1               True   0.089926  0.624405   
1        4        11    0               True   0.067761  0.561488   
         55       5     0               True   1.194624  0.608719   
2        26       7     0               True   0.145852  0.648713   
...                                      ...        ...       ...   
9992     53       3     1               True   0.125378  0.733339   
9993     42       10    1               True   1.248511  0.733077   
9995     6        3     1               True   0.086470  0.706942   
         35       5     0               True   1.104667  0.717255   
9996     2        13    2               True   0.137584  0.664276   

                                       is_not_clear_cosmic n_prong n_trk  \
                                                                           
file_idx __ntuple entry rec.slc..index                                     
0        3        4     3                             True       2     2   
         36       6     1                             True       2     2   
1        4        11    0                             True       2     2   
         55       5     0                             True       2     2   
2        26       7     0                             True       2     2   
...                                                    ...     ...   ...   
9992     53       3     1                             True       2     2   
9993     42       10    1                             True       2     2   
9995     6        3     1                             True       2     2   
         35       5     0                             True       2     2   
9996     2        13    2                             True       2     2   

                                       n_proton n_shower tmatch_idx  \
                                                                      
file_idx __ntuple entry rec.slc..index                                
0        3        4     3                     0        0          0   
         36       6     1                     0        0          1   
1        4        11    0                     0        0          0   
         55       5     0                     0        0          0   
2        26       7     0                     0        0          1   
...                                         ...      ...        ...   
9992     53       3     1                     0        0          0   
9993     42       10    1                     0        0          1   
9995     6        3     1                     0        0          0   
         35       5     0                     0        0          0   
9996     2        13    2                     0        0          0   

                                       tmatch_eff  ...  \
                                                   ...   
file_idx __ntuple entry rec.slc..index             ...   
0        3        4     3                0.850429  ...   
         36       6     1                0.884164  ...   
1        4        11    0                0.968936  ...   
         55       5     0                0.755416  ...   
2        26       7     0                0.857026  ...   
...                                           ...  ...   
9992     53       3     1                0.702602  ...   
9993     42       10    1                0.905247  ...   
9995     6        3     1                0.925048  ...   
         35       5     0                0.869015  ...   
9996     2        13    2                0.880691  ...   

                                       reinteractions_proton_Geant4            \
                                                           univ_990  univ_991   
file_idx __ntuple entry rec.slc..inde

In [13]:
mcnu_df

true_E_nu true_p_mu true_p_pi  \
                                                                         
file_idx __ntuple entry rec.mc.nu..index                                 
3        63       11    0                 1.706439  0.979571  0.707553   
7        67       6     0                 1.053418  0.726049  0.287649   
11       16       5     0                 1.107123  0.887457  0.161428   
         37       9     0                 1.007266  0.275232  0.698645   
12       3        1     1                 0.731182  0.327431  0.361404   
...                                            ...       ...       ...   
9991     20       14    1                 1.650084  1.000668  0.628544   
9992     29       4     0                 0.773652  0.367518  0.323879   
         53       12    1                 1.111488  0.771026  0.262806   
9993     63       12    0                 1.232724  0.882134  0.314726   
9994     52       8     0                 2.197856  1.865019  0.298862   

                                         true_cos_theta_mu true_cos_theta_pi  \
                                                                               
file_idx __ntuple entry rec.mc.nu..index                                       
3        63       11    0                         0.994755          0.986379   
7        67       6     0                         0.999657          0.981709   
11       16       5     0                         0.997335          0.584882   
         37       9     0                         0.897589          0.952007   
12       3        1     1                         0.999983          0.984496   
...                                                    ...               ...   
9991     20       14    1                         0.997429          0.995284   
9992     29       4     0                         0.977397          0.838347   
         53       12    1                         0.997293          0.432034   
9993     63       12    0                         0.992556          0.778005   
9994     52       8     0                         0.999055          0.957266   

                                            true_t nuint_categ ind  \
                                                                     
file_idx __ntuple entry rec.mc.nu..index                             
3        63       11    0                 0.029576           1   0   
7        67       6     0                 0.004504           1   0   
11       16       5     0                 0.021260           1   0   
         37       9     0                 0.023696           1   0   
12       3        1     1                 0.006166           1   1   
...                                            ...         ...  ..   
9991     20       14    1                 0.000994           1   1   
9992     29       4     0                 0.027338           1   0   
         53       12    1                 0.085544           1   1   
9993     63       12    0                 0.026059           1   0   
9994     52       8     0                 0.006351           1   0   

                                         GENIEReWeight_SBN_v1_multisigma_ZExpA1CCQE  \
                                                                                ps1   
file_idx __ntuple entry rec.mc.nu..index                                              
3        63       11    0                                                       1.0   
7        67       6     0                                                       1.0   
11       16       5     0                                                       1.0   
         37       9     0                                                       1.0   
12       3        1     1                                                       1.0   
...                                                                             ...   
9991     20       14    1                                                       1.0   
9992     29       4     0       

In [14]:
hdr_df

pot  first_in_subrun  ismc   run  subrun  \
file_idx __ntuple entry                                                      
0        0        0      8.819115e+14                1     1   846      66   
                  1      0.000000e+00                0     1   846      66   
                  2      0.000000e+00                0     1   846      66   
                  3      0.000000e+00                0     1   846      66   
                  4      0.000000e+00                0     1   846      66   
...                               ...              ...   ...   ...     ...   
9999     73       8      0.000000e+00                0     1  7869      17   
                  9      0.000000e+00                0     1  7869      17   
                  10     0.000000e+00                0     1  7869      17   
                  11     0.000000e+00                0     1  7869      17   
                  12     0.000000e+00                0     1  7869      17   

                         ngenevt  evt  proc  cluster  fno  noffbeambnb  
file_idx __ntuple entry                                                 
0        0        0           50    3    -1       -1    0          0.0  
                  1           50    4    -1       -1    0          0.0  
                  2           50    6    -1       -1    0          0.0  
                  3           50   10    -1       -1    0          0.0  
                  4           50   11    -1       -1    0          0.0  
...                          ...  ...   ...      ...  ...          ...  
9999     73       8           50   37    -1       -1    0          0.0  
                  9           50   42    -1       -1    0          0.0  
                  10          50   47    -1       -1    0          0.0  
                  11          50   49    -1       -1    0          0.0  
                  12          50   50    -1       -1    0          0.0  

[9935465 rows x 11 columns]

In [15]:
histpotdf_df

TotalPOT
file_idx __ntuple entry              
0        0        0      8.819115e+14
         1        0      8.943580e+14
         2        0      6.622154e+14
         3        0      7.531234e+14
         4        0      7.728901e+14
...                               ...
9999     69       0      6.605002e+14
         70       0      7.772624e+14
         71       0      7.334867e+14
         72       0      6.193842e+14
         73       0      6.394750e+14

[749258 rows x 1 columns]

In [16]:
histgenevtdf_df

TotalGenEvents
file_idx __ntuple entry                
0        0        0                50.0
         1        0                50.0
         2        0                50.0
         3        0                50.0
         4        0                50.0
...                                 ...
9999     69       0                50.0
         70       0                50.0
         71       0                50.0
         72       0                50.0
         73       0                50.0

[749258 rows x 1 columns]

In [17]:
hdr_df.pot.sum()

5.4017982e+20

In [18]:
histpotdf_df.TotalPOT.sum()

5.4017996516021535e+20

In [19]:
histgenevtdf_df.TotalGenEvents.sum()

37462900.0

### Save the df

In [20]:
with pd.HDFStore('cohpi_aurora_5p4e20_systs_no_dirz.df') as store:
    store.put('evt', evt_df, format='fixed')
    store.put('mcnu', mcnu_df, format='fixed')
    store.put('pot', pot_df, format='fixed')
    store.put('hdr', hdr_df, format='fixed')
    store.put('histpotdf', histpotdf_df, format='fixed')
    store.put('histgenevtdf', histgenevtdf_df, format='fixed')

/home/sungbino/py/cohpi/lib64/python3.9/site-packages/tables/attributeset.py:457: NaturalNameWarning: object name is not a valid Python identifier: 'axis1_namerec.slc..index'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$``; you will not be able to use natural naming to access this object; using ``getattr()`` will still work, though
  check_attribute_name(name)
/tmp/ipykernel_17738/1577787551.py:2: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->boolean,key->block5_values] [items->MultiIndex([(  'short_dirz_pass', ''),
            ('is_muon_contained', ''),
            ( 'is_cpi_contained', '')],
           )]

  store.put('evt', evt_df, format='fixed')
/home/sungbino/py/cohpi/lib64/python3.9/site-packages/tables/attributeset.py:457: NaturalNameWarning: object name is not a valid Python identifier: 'axis1_namerec.mc.nu..index'; it does not match the pattern ``^[a-zA-Z_][a-zA-Z0-9_]*$